In [0]:
from pyspark.sql import functions as F

bronze_path = "/Volumes/workspace/default/filestore/day12/bronze/orders"
silver_path = "/Volumes/workspace/default/filestore/day12/silver/orders"

bronze_df = spark.read.format("delta").load(bronze_path)

display(bronze_df)

order_id,customer_id,order_date,amount
101,1,2026-09-20,500.0
102,1,2026-09-21,1200.0
103,2,2026-09-21,800.0
104,3,2026-09-22,2500.0
105,2,2026-09-23,1500.0


In [0]:
silver_df = (
    bronze_df
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("amount", F.col("amount").cast("double"))
)

In [0]:
silver_vaild_df = silver_df.filter(
    (F.col("order_id").isNotNull()) &
    (F.col("customer_id").isNotNull()) &
    (F.col("order_date").isNotNull()) &
    (F.col("amount").isNotNull()) &
    (F.col("amount") >= 0)
)

silver_vaild_df = silver_vaild_df.dropDuplicates(["order_id"])

In [0]:
silver_vaild_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

print("Silver transformation Completed")

Silver transformation Completed


In [0]:
final_silver_df =  spark.read.format("delta").load(silver_path)

print("Silver record count:", final_silver_df.count())

display(final_silver_df)

Silver record count: 5


order_id,customer_id,order_date,amount
102,1,2026-09-21,1200.0
104,3,2026-09-22,2500.0
101,1,2026-09-20,500.0
103,2,2026-09-21,800.0
105,2,2026-09-23,1500.0
